In [ ]:
import pandas as pd
import re
import os
import traceback

def clean_text(text):
    if not isinstance(text, str):
        return str(text) if pd.notna(text) else ""
    
    text = text.replace('\\n', ' ').replace('\\r', ' ')
    text = text.replace('\n', ' ').replace('\r', ' ')

    text = re.sub(r'\s*https?\s*:\s*[\\/]+\S+', '', text, flags=re.IGNORECASE)
    
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def process_and_compare(txt_input, excel_target, output_file):
    try:
        print(f"--------------------------------------------------------")
        print(f"Processing Task: {output_file}")
        print(f"[1/3]: Loading TXT {os.path.basename(txt_input)}")
        
        df_txt = pd.read_csv(txt_input, sep='\t', dtype={'tweetId': str}, on_bad_lines='skip')
        
        if 'tweetId' in df_txt.columns and 'tweetText' in df_txt.columns:
            df_txt = df_txt[['tweetId', 'tweetText']]
            df_txt.columns = ['id', 'text']
        else:
            print("error：no 'tweetId' or 'tweetText' in txt file")
            return

        print(" -> cleaning text...")
        df_txt['text'] = df_txt['text'].apply(clean_text)
        
        df_txt = df_txt.drop_duplicates(subset=['id'])
        print(f" -> {len(df_txt)} txt samples loaded")

        print(f"[2/3]: Loading Excel {os.path.basename(excel_target)}")
        
        df_base = pd.read_excel(excel_target, dtype={'id': str})
        
        if 'id' not in df_base.columns:
            raise ValueError("no 'id' in excel file")
            
        if 'text' in df_base.columns:
            df_base = df_base.drop(columns=['text'])
        
        print(f" -> {len(df_base)} base samples loaded")

        print("[3/3] Merging and Saving...")

        merged_df = pd.merge(
            df_base, 
            df_txt, 
            on='id', 
            how='left'
        )

        merged_df['text'] = merged_df['text'].fillna('')

        priority_cols = ['id', 'text']
        other_cols = [c for c in merged_df.columns if c not in priority_cols]
        final_df = merged_df[priority_cols + other_cols]
        
        final_df.to_excel(output_file, index=False)
        
        print(f"Success! Saved to: {output_file}")
        print(f"Total rows: {len(final_df)}")
        print(f"--------------------------------------------------------\n")
        
    except Exception as e:
        print(f"\nError processing {output_file}: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    # Task tuple format: (raw_dataset_txt, llm_description_excel, output_cleaned_merged_excel)
    tasks = [
        (
            r"test_tw.txt",          # Raw text data (tweet.txt) downloaded from the Twitter15 dataset (Test split)
            r"twitter_test_news.xlsx",  # Provided spreadsheet containing LLM-generated objective news event descriptions (Test split)
            r"tw15_test.xlsx"        # Target output path for the finalized, cleaned, and merged data (Test split)
        ),
        (
            r"train_tw.txt",         # Raw text data (tweet.txt) downloaded from the Twitter15 dataset (Train split)
            r"twitter_train_news.xlsx", # Provided spreadsheet containing LLM-generated objective news event descriptions (Train split)
            r"tw15_train.xlsx"       # Target output path for the finalized, cleaned, and merged data (Train split)
        )
    ]
    print(">>> Batch Processing Started <<<\n")

    for txt_path, excel_path, result_path in tasks:
        if os.path.exists(txt_path) and os.path.exists(excel_path):
            process_and_compare(txt_path, excel_path, result_path)
        else:
            print(f"Skipping task for {result_path}: Input file not found.")
            if not os.path.exists(txt_path):
                print(f" - Missing: {txt_path}")
            if not os.path.exists(excel_path):
                print(f" - Missing: {excel_path}")
            print("--------------------------------------------------------\n")

    print(">>> Batch Processing Finished <<<")